# Multiprocessing in Python â€“ Recap and Pitfalls

## What is Multiprocessing?

**Multiprocessing** is a technique that allows a program to run **multiple processes in parallel**, each with its own Python interpreter and memory space.
It is especially useful for **CPU-bound tasks**, where computations can be distributed across multiple CPU cores.

In Python, multiprocessing is commonly implemented using the built-in `multiprocessing` module.

## Basic Concurrency Concepts in Python

Before working with multiprocessing, it is important to understand some **fundamental terms** used in concurrent and parallel programming.

---

### Process

A **process** is an independent running instance of a program.

#### Characteristics:

* Has its **own memory space**
* Has at least one thread
* Is isolated from other processes
* More expensive to create than threads

---

### Thread

A **thread** is a lightweight unit of execution **inside a process**.

#### Characteristics:

* Threads **share the same memory**
* Faster to create than processes
* Easier to share data
* Risk of race conditions

---

### Pool

A **pool** is a collection of worker threads or processes that execute tasks from a queue.

#### Characteristics:

* Avoids repeatedly creating and destroying workers
* Controls the level of parallelism
* Simplifies task distribution

---

### Task

A **task** is a unit of work submitted to a thread or process.

# PyTorch Multiprocessing 

When models process **large datasets**, perform **heavy preprocessing**, or coordinate **multiple GPUs**, multiprocessing becomes a key performance tool.

---

## Why PyTorch Has Its Own Multiprocessing Module

PyTorch provides `torch.multiprocessing`, which is a **PyTorch-aware wrapper** around Pythonâ€™s built-in `multiprocessing`.

### Key advantages over standard multiprocessing:

* Efficient **tensor sharing** between processes
* Native support for **CUDA tensors**
* Reduced overhead when passing data
* Safe handling of PyTorchâ€™s internal memory model

---

## Basic PyTorch Multiprocessing Example (Shared Tensor)

Letâ€™s start with a minimal example showing **multiple processes modifying a shared tensor**.

```python
# IMPORTANT:
# This code must be executed from a Python file (.py).
# It will NOT run correctly in interactive environments
# such as Jupyter Notebook, because
# torch.multiprocessing requires the __main__ guard
# to safely spawn child processes.

import torch
import torch.multiprocessing as mp

def worker_function(rank, shared_tensor):
    """
    rank:          unique ID of the process
    shared_tensor: tensor stored in shared memory
    """
    print(f"Process {rank} started")
    shared_tensor[rank] += 1  # In-place modification of shared tensor

if __name__ == "__main__":
    # Create a tensor in shared memory
    tensor = torch.zeros(4)
    tensor.share_memory_()

    processes = []

    # Spawn 4 independent processes
    for rank in range(4):
        p = mp.Process(
            target=worker_function,
            args=(rank, tensor)
        )
        p.start()
        processes.append(p)

    # Wait for all processes to finish
    for p in processes:
        p.join()

    print("Updated Tensor:", tensor)
```

### What is happening step by step?

1. `tensor.share_memory_()`

   * Moves the tensor into shared memory
   * Allows all processes to access the same data

2. `mp.Process(...)`

   * Creates a new OS-level process
   * Each process runs `worker_function`

3. `shared_tensor[rank] += 1`

   * Each process updates a different element
   * No race condition in this case

4. Final output:

```text
Updated Tensor: tensor([1., 1., 1., 1.])
```

## Example: Is it usefull?

```python
import math
import time
import multiprocessing as mp
import os


# CPU-heavy function (pure Python)
def heavy_work(n: int) -> float:
    acc = 0.0
    for i in range(1, n):
        acc += math.sqrt(i) * math.sin(i)
    return acc


# Worker wrapper (for logging)
def worker_task(n: int) -> float:
    pid = os.getpid()
    print(f"[PID {pid}] start work")
    result = heavy_work(n)
    print(f"[PID {pid}] end work")
    return result


# Sequential baseline
def run_sequential(tasks: int, n: int) -> float:
    print("\n=== SEQUENTIAL ===")
    start = time.time()

    results = []
    for _ in range(tasks):
        results.append(heavy_work(n))

    elapsed = time.time() - start
    print(f"Sequential time: {elapsed:.2f} s")
    return elapsed


# Multiprocessing version
def run_multiprocessing(tasks: int, n: int, workers: int) -> float:
    print(f"\n=== MULTIPROCESSING ({workers} workers) ===")
    start = time.time()

    with mp.Pool(processes=workers) as pool:
        results = pool.map(worker_task, [n] * tasks)

    elapsed = time.time() - start
    print(f"Multiprocessing time: {elapsed:.2f} s")
    return elapsed


# Main entry point 
if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)

    TASKS = 8          # number of independent jobs
    N = 500_000        # cost of a single job
    WORKERS = 4        # number of processes

    t_seq = run_sequential(TASKS, N)
    t_mp = run_multiprocessing(TASKS, N, WORKERS)

    print("\n=== SUMMARY ===")
    print(f"Sequential:      {t_seq:.2f} s")
    print(f"Multiprocessing:{t_mp:.2f} s")
    print(f"Speedup:         {t_seq / t_mp:.2f}x")
```


# Setting Up Multiprocessing in PyTorch

Before using multiprocessing in real PyTorch workflows, you must **choose the correct process start method**.
This choice affects **performance, stability, GPU compatibility, and platform support**.

To understand *why* these methods behave differently, we first need to explain two key concepts:

* the **Python interpreter**
* the **main module (`__main__`)**

---

## What Is the Python Interpreter?

The **Python interpreter** is the program that executes Python code.

### What the interpreter does:

* reads `.py` files
* converts code to bytecode
* executes instructions step by step
* manages memory and objects
* keeps the **program state**

### Program state includes:

* variables and tensors
* imported modules
* random number generator state
* open files and sockets
* CUDA contexts (in PyTorch)

> **Important:**
> Your script is *not* the interpreter.
> The interpreter is the **engine**, the script is just **instructions**.

---

## What Is the Main Module (`__main__`)?

The **main module** is the Python file that is **executed directly**.

Example:

```bash
python train.py
```
---

## The `if __name__ == "__main__":` Guard

```python
if __name__ == "__main__":
    start_processes()
```

### What this guard does:

* Code inside runs **only when the file is executed directly**
* Code is skipped when the file is imported

### Why this matters for multiprocessing:

Some start methods **re-import the main module**.
Without this guard, child processes may accidentally re-run the same code, leading to:

* infinite process spawning
* crashes
* frozen programs

---

## Why Start Methods Matter

When a new process is created, the key question is:

> **Does the new process inherit the parent interpreterâ€™s state,
> or does it start with a fresh interpreter?**

PyTorch supports three start methods:

* `spawn`
* `fork`
* `forkserver`

Each answers this question differently.

Choosing the wrong one can lead to **silent bugs, crashes, or CUDA errors**.

---

## Spawn Method (Recommended & Cross-Platform)

The **spawn** method starts a **fresh Python interpreter** for each process.

### What happens internally:

```text
New process
â†’ New Python interpreter
â†’ Import main module
â†’ Execute top-level code
â†’ Run target function
```

### When to use:

* **Windows (mandatory)**
* CUDA / GPU workloads
* When safety and correctness matter more than raw speed

### Key properties:

* Clean process state
* No inherited memory
* Requires `__main__` guard
* Slightly slower startup
* Most stable option

### Example: Using `spawn`

```python
import torch.multiprocessing as mp

def task(rank):
    print(f"Process {rank} says hello!")

if __name__ == "__main__":
    mp.set_start_method("spawn")

    processes = []
    for i in range(4):
        p = mp.Process(target=task, args=(i,))
        p.start()
        processes.append(p)

    for p in processes:
        p.join()
```

---

## Fork Method (Fast but Risky)

The **fork** method duplicates the current process **without starting a new interpreter**.

### What happens internally:

```text
Parent interpreter
â†’ fork()
â†’ Child = memory copy of parent
```

### When to use:

* Linux or macOS only
* CPU-only workloads
* Performance-critical pipelines
* Only if you fully understand the risks

### Key properties:

* Very fast startup
* Inherits entire interpreter state
* No re-import of main module
* Dangerous with CUDA
* Harder to debug

### Example: Using `fork`

```python
import torch.multiprocessing as mp

def task(rank):
    print(f"Process {rank} says hello!")

if __name__ == "__main__":
    mp.set_start_method("fork")

    processes = []
    for i in range(4):
        p = mp.Process(target=task, args=(i,))
        p.start()
        processes.append(p)

    for p in processes:
        p.join()
```

### Example: `spawn` vs `fork`

```python
import multiprocessing as mp
import random

random.seed(42)

def worker():
    print(random.random())

if __name__ == "__main__":
    mp.set_start_method("spawn") 
    # mp.set_start_method("fork") # try both

    for _ in range(3):
        p = mp.Process(target=worker)
        p.start()
        p.join()
```

---

## Forkserver Method (Controlled and Stable)

The **forkserver** method uses a **clean server process** as a template.

### What happens internally:

```text
Forkserver (clean interpreter)
â†’ fork
â†’ Child process
```

### When to use:

* Large or sensitive workloads
* Long-running pipelines
* When `fork` causes crashes
* When you want controlled memory inheritance

### Key properties:

* More stable than `fork`
* Faster than `spawn`
* Available only on UNIX
* Good safetyâ€“performance compromise

### Example: Using `forkserver`

```python
import torch.multiprocessing as mp

def task(rank):
    print(f"Process {rank} says hello!")

if __name__ == "__main__":
    mp.set_start_method("forkserver")

    processes = []
    for i in range(4):
        p = mp.Process(target=task, args=(i,))
        p.start()
        processes.append(p)

    for p in processes:
        p.join()
```

# Data Sharing Between Processes in PyTorch

When working with multiprocessing, the hardest part is usually **not starting processes**, but **sharing data between them safely and efficiently**.

This is where PyTorch stands out: it provides **native shared-memory support for tensors**, which avoids expensive data copying and enables high-performance multiprocessing workflows.

---

## Shared Memory in PyTorch

By default, each process has its **own private memory**.
If a tensor is not explicitly shared, **every process receives its own copy**.

PyTorch solves this with **shared tensors**.

### Key concept:

> A shared tensor exists **once in memory**, but can be accessed and modified by multiple processes.

To enable this, PyTorch provides the method:

```python
tensor.share_memory_()
```

This moves the tensor into shared memory and allows all child processes to see the same data.

---

## Using Shared Tensors (`share_memory_()`)

Letâ€™s look at a simple example where each process updates a different part of the same tensor.

```python
import torch
import torch.multiprocessing as mp

def worker_function(rank, shared_tensor):
    """
    Each process modifies a unique index in the shared tensor.
    """
    print(f"Process {rank} modifying shared tensor")
    shared_tensor[rank] += rank  # Safe: each process writes to its own index

if __name__ == "__main__":
    mp.set_start_method("spawn")

    # Create a tensor and move it to shared memory
    shared_tensor = torch.zeros(4)
    shared_tensor.share_memory_()

    processes = []

    # Spawn 4 processes
    for rank in range(4):
        p = mp.Process(
            target=worker_function,
            args=(rank, shared_tensor)
        )
        p.start()
        processes.append(p)

    # Wait for all processes to finish
    for p in processes:
        p.join()

    print("Updated shared tensor:", shared_tensor)
```

### Step-by-step explanation:

1. `torch.zeros(4)`
   Creates a tensor in the main process.

2. `shared_tensor.share_memory_()`
   Moves the tensor into shared memory.

3. Each process receives a **reference** to the same tensor.

4. Each process updates a **unique index** (`rank`), avoiding conflicts.

### Output:

```text
Updated shared tensor: tensor([0., 1., 2., 3.])
```

---

## Why Shared Tensors Are Efficient

Without shared memory:

* Tensors must be **serialized**
* Copied between processes
* Deserialized again

With shared memory:

* No copying
* No serialization
* Near-zero communication overhead

This is why shared tensors are essential for:

* Parallel data preprocessing
* Shared buffers
* Multi-process data pipelines

# Implementing Custom PyTorch Multiprocessing Workflows

Now that you understand **process creation** and **shared memory**, we can build **custom multiprocessing pipelines** in PyTorch.

This is where multiprocessing becomes truly powerful: instead of relying only on built-in abstractions, you can design **tailored parallel workflows** for your project.

---

## Example Use Case: Parallel Data Preprocessing

Imagine a data-heavy task such as **image preprocessing**:

* resizing
* normalization
* augmentation

Doing this sequentially can become a major bottleneck. Multiprocessing allows us to **process multiple samples at the same time**.

---

## Parallel Image Preprocessing with Shared Tensors

Each process:

* loads one image
* applies transformations
* writes the result to a shared output tensor


```python
import torch
import torch.multiprocessing as mp
from PIL import Image
import torchvision.transforms as transforms

def process_image(rank, image_path, output_tensor):
    """
    Worker function executed in a separate process.

    rank:         index identifying the process
    image_path:   path to the image to process
    output_tensor: shared tensor where results are stored
    """
    image = Image.open(image_path)

    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor()
    ])

    output_tensor[rank] = transform(image)
    print(f"Process {rank} processed image")

if __name__ == "__main__":
    mp.set_start_method("spawn")

    # Paths to input images
    image_paths = [
        "image1.jpg",
        "image2.jpg",
        "image3.jpg",
        "image4.jpg"
    ]

    # Shared output tensor: (batch, channels, height, width)
    output_tensor = torch.zeros((4, 3, 128, 128))
    output_tensor.share_memory_()

    processes = []

    # Spawn one process per image
    for rank, image_path in enumerate(image_paths):
        p = mp.Process(
            target=process_image,
            args=(rank, image_path, output_tensor)
        )
        p.start()
        processes.append(p)

    # Wait for all processes to finish
    for p in processes:
        p.join()

    print("Processed images stored in tensor:", output_tensor)
```

### Why this works well:

* Each process writes to a **unique index**
* No locking required
* Shared tensor avoids copying data
* Preprocessing scales with CPU cores

---

## Designing Worker Functions (Best Practices)

A **well-designed worker function** should:

* Perform **one clearly defined task**
* Accept:

  * a process identifier (`rank`)
  * input data
  * an output location
* Avoid global state
* Minimize side effects

### Recommended worker structure:

```text
worker(rank, input, output, optional_sync)
```

This design:

* Reduces bugs
* Avoids race conditions
* Makes debugging easier

---

## Fine-Grained Parallelization Example

Below, each process applies the same model sub-module to different data.

```python
import torch
import torch.nn as nn
import torch.multiprocessing as mp

class SubModule(nn.Module):
    def forward(self, x):
        return x ** 2  # Simple operation for demonstration

def parallel_task(rank, model, data, output_tensor):
    """
    Each process computes one part of the output.
    """
    output_tensor[rank] = model(data)
    print(f"Process {rank} finished computation")

if __name__ == "__main__":
    mp.set_start_method("spawn")

    data = torch.ones(4)
    output_tensor = torch.zeros(4)
    output_tensor.share_memory_()

    model = SubModule()

    processes = []

    for rank in range(4):
        p = mp.Process(
            target=parallel_task,
            args=(rank, model, data[rank], output_tensor)
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    print("Parallel model outputs:", output_tensor)
```

## Multiprocessing Inside `DataLoader`

PyTorch already uses multiprocessing internally via `DataLoader`.

### Example: Parallel batch loading

```python
import torch
from torch.utils.data import DataLoader, TensorDataset, get_worker_info
import os
import time

def collate_fn(batch):
    info = get_worker_info()
    pid = os.getpid()
    wid = info.id if info is not None else "main"
    print(f"[PID {pid}] worker_id={wid}")
    time.sleep(0.2)
    return torch.utils.data.default_collate(batch)

def main():
    data = torch.randn(1000, 10)
    labels = torch.randint(0, 2, (1000,))
    dataset = TensorDataset(data, labels)

    dataloader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=True,
        num_workers=4,
        collate_fn=collate_fn
    )

    for batch_idx, (x, y) in enumerate(dataloader):
        print(f"===> Batch {batch_idx} received")
        if batch_idx == 5:
            break

if __name__ == "__main__":
    main()
```

### What `num_workers=4` does:

* Spawns 4 worker processes
* Loads batches in parallel
* Overlaps data loading with GPU computation
* Significantly improves training throughput

# Synchronizing Data and Model Parameters Across Processes

When multiple processes run in parallel, **coordination becomes critical**.
Just like multiple people editing the same document need rules to avoid conflicts, processes need **synchronization mechanisms** to safely access shared resources.

Without synchronization, your program may:

* produce incorrect results
* behave nondeterministically
* silently corrupt data

---

## Locking and Synchronization

A **lock** ensures that **only one process at a time** can access a shared resource.

### Why locks are needed

Without locking:

* Two processes may update the same value simultaneously
* Updates may overwrite each other
* Final results become unpredictable

This situation is known as a **race condition**.

---

## Example: Safely Updating a Shared Counter

Letâ€™s look at a classic synchronization example: **incrementing a shared counter**.

```python
import torch.multiprocessing as mp

def increment_counter(counter, lock):
    """
    Worker function that safely increments a shared counter.
    """
    for _ in range(1000):
        with lock:
            counter.value += 1

if __name__ == "__main__":
    mp.set_start_method("spawn")

    # Shared integer counter
    counter = mp.Value('i', 0)

    # Lock to synchronize access
    lock = mp.Lock()

    # Create multiple processes
    processes = [
        mp.Process(target=increment_counter, args=(counter, lock))
        for _ in range(4)
    ]

    for p in processes:
        p.start()
    for p in processes:
        p.join()

    print("Final counter value:", counter.value)
```

### What is happening?

* `mp.Value('i', 0)`
  Creates a shared integer stored in shared memory.

* `with lock:`
  Ensures that only one process modifies the counter at a time.

* Expected output:

```text
Final counter value: 4000
```

### What if we remove the lock?

* The final value will vary
* Some increments will be lost
* This demonstrates a **race condition**

# Parallel Training of Neural Networks

When training modern neural networks, datasets and models often become too large to process efficiently on a single CPU or GPU. To reduce training time, we **train models in parallel** using multiple workers.

There are two main ways to parallelize training:

---

## Asynchronous Training (Parameter Server style)

In asynchronous training:

* multiple workers compute gradients independently
* updates are applied **without waiting** for other workers
* workers may use **stale parameters**

### Advantages

* high hardware utilization
* no waiting between workers

### Disadvantages

* gradients may be inconsistent
* convergence is harder to analyze
* training may become unstable

Asynchronous training is rarely used in modern deep learning frameworks **for supervised learning**, but appears in some large-scale or reinforcement learning systems.

---

## Synchronous Data Parallel Training (Recommended)

In synchronous data parallel training:

* each worker processes a different mini-batch
* gradients are **synchronized and averaged**
* all models remain **identical after each update**

This is the approach implemented by **DistributedDataParallel (DDP)** in PyTorch.

---

## What DistributedDataParallel (DDP) Does

At a high level:

1. Each process has its **own copy of the model**
2. Each process computes gradients on its **local mini-batch**
3. During backpropagation:

   * gradients are **communicated across processes**
   * an **all-reduce** operation averages them
4. Each process updates its model using the same averaged gradients

As a result:

* training is mathematically equivalent to training with a **larger batch size**
* no explicit locking or shared memory is required
* training is stable and reproducible

---

## Why We Do NOT Use Locks or Shared Memory

A common beginner mistake is trying to:

* share a single model in memory
* protect updates using locks

This approach:

* does not scale
* is error-prone
* breaks gradient assumptions

DDP avoids these issues by using **message passing**, not shared memory.

---

## When Should You Use DDP?

Use DistributedDataParallel when:

* your model or dataset is large
* you want to scale training across CPUs or GPUs
* you need stable and well-understood convergence behavior

Do **not** use DDP when:

* your model is tiny
* training fits easily on one device
* parallel overhead dominates computation

```python
import torch
import torch.nn as nn
import torch.multiprocessing as mp
import torch.distributed as dist

# Simple neural network model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 1)

    def forward(self, x):
        return self.fc(x)

def train(rank, world_size, num_epochs):
    dist.init_process_group(
        backend="gloo",
        init_method="tcp://127.0.0.1:29500",
        rank=rank,
        world_size=world_size
    )

    model = SimpleModel()
    model = nn.parallel.DistributedDataParallel(model)
    model.train()

    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()

    for epoch in range(num_epochs):
        data = torch.randn(8, 10)   # batch
        target = torch.randn(8, 1)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        if rank == 0:
            print(f"Epoch {epoch} done")

    dist.destroy_process_group()

if __name__ == "__main__":
    world_size = 4     # Number of parallel processes
    num_epochs = 5     # Number of epochs

    mp.spawn(
        train,
        args=(world_size, num_epochs),
        nprocs=world_size
    )
```

# Error Handling, Debugging, and Optimization in PyTorch Multiprocessing

**Debugging multiprocessing code is hard**.
Errors may appear only sometimes, logs may interleave, and crashes can happen far from their real cause.

With **structured logging**, **safe communication**, and **profiling**, multiprocessing becomes much more manageable.

---

## Logging and Error Management in Multiprocessing

When multiple processes run simultaneously, `print()` is often not enough.
Instead, each process should **log its actions explicitly**.

### Why logging matters:

* Identifies **which process failed**
* Preserves execution order
* Makes errors reproducible
* Essential for large-scale experiments

---

## Example: Logging Errors from Multiple Processes

```python
import logging
import torch.multiprocessing as mp

def worker_function(rank):
    logging.info(f"Process {rank} starting.")
    try:
        # Simulate work with a possible error
        result = 100 / rank  # Division by zero for rank = 0
    except Exception as e:
        logging.error(f"Error in process {rank}: {e}")
    finally:
        logging.info(f"Process {rank} completed.")

if __name__ == "__main__":
    mp.set_start_method("spawn")

    logging.basicConfig(
        level=logging.INFO,
    )

    processes = [
        mp.Process(target=worker_function, args=(i,))
        for i in range(4)
    ]

    for p in processes:
        p.start()
    for p in processes:
        p.join()
```